# Discovery, Validation, And Artifact Inspection

This notebook uses the shared Drive dataset as a read-only source, uses local Colab artifacts for reliability, reuses the training notebook runtime config when available, and exports the finished discovery/validation outputs to a clearly separate Drive folder.

In [ ]:
# Configuration — edit these values, then Run All
# If colab_runtime_experiment.yaml already exists (e.g. from a same-session
# training run), NOTEBOOK_DATASET is ignored and that config is used directly.

USE_FULL_DATASET = False
EXPERIENCE_LEVEL = 'Familiar'
MAX_SESSIONS = 10
NOTEBOOK_STEP_LOG_EVERY = 16
DISCOVERY_SPLIT_CANDIDATES = ['discovery', 'valid', 'test']

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import json
import pandas as pd
from datetime import datetime

drive.mount('/content/drive')
repo_root = Path('/content/predictive_circuit_coding')
github_repo_url = 'https://github.com/jacobposchl/predictive_circuit_coding'

# Shared folder from the storage-heavy Drive account: read dataset from here.
shared_drive_root = Path('/content/drive/MyDrive/pcc_runs')
drive_data_root = shared_drive_root / 'data' / 'allen_visual_behavior_neuropixels'

# Keep exported Colab outputs in a clearly different folder name so they do not get
# confused with the source dataset bundle.
drive_export_root = Path('/content/drive/MyDrive/pcc_colab_outputs')
drive_export_root.mkdir(parents=True, exist_ok=True)

print('Repo root:', repo_root)
print('GitHub repo:', github_repo_url)
print('Shared dataset root:', drive_data_root)
print('Drive export root:', drive_export_root)

In [ ]:
if not repo_root.exists():
    !git clone {github_repo_url} {repo_root}
%cd {repo_root}
# Keep Colab installs minimal to avoid restart prompts from preloaded widget packages.
!pip install -e .

In [ ]:
repo_dataset_root = repo_root / 'data' / 'allen_visual_behavior_neuropixels'
repo_artifacts_root = repo_root / 'artifacts'

assert drive_data_root.exists(), f'Missing Drive dataset bundle: {drive_data_root}'

repo_dataset_root.parent.mkdir(parents=True, exist_ok=True)
if repo_dataset_root.exists() or repo_dataset_root.is_symlink():
    if repo_dataset_root.is_symlink():
        repo_dataset_root.unlink()
    else:
        shutil.rmtree(repo_dataset_root)
repo_dataset_root.symlink_to(drive_data_root, target_is_directory=True)

if not repo_artifacts_root.exists():
    repo_artifacts_root.mkdir(parents=True, exist_ok=True)

print('Linked dataset into repo:', repo_dataset_root)
print('Using local artifact directory:', repo_artifacts_root)

In [ ]:
import subprocess
import yaml
from predictive_circuit_coding.utils import (
    NotebookDatasetConfig,
    NotebookStageReporter,
    output_indicates_missing_positive_labels,
    prepare_notebook_runtime_context,
    resolve_notebook_checkpoint,
    restore_latest_exported_artifacts,
    run_streaming_command,
    verify_paths_exist,
)

NOTEBOOK_DATASET = NotebookDatasetConfig(
    use_full_dataset=USE_FULL_DATASET,
    experience_level=EXPERIENCE_LEVEL,
    max_sessions=MAX_SESSIONS,
)

reporter = NotebookStageReporter(name='colab-discover', expected_duration='1-5 minutes for debug runs, longer for larger discovery passes')
reporter.banner('Discovery And Validation', subtitle='Frozen-token discovery, validation controls, inspection, and export')

base_experiment_config = repo_root / 'configs' / 'pcc' / 'predictive_circuit_coding_base.yaml'
data_config = repo_root / 'configs' / 'pcc' / 'allen_visual_behavior_neuropixels_local.yaml'
runtime_experiment_config = repo_root / 'colab_runtime_experiment.yaml'
checkpoint_dir = repo_root / 'artifacts' / 'checkpoints'
summary_path = repo_root / 'artifacts' / 'training_summary.json'
selection_output_name = 'unknown'
selected_session_count = None
restored_export_run = None

if not summary_path.exists() and not checkpoint_dir.exists():
    restored_export_run = restore_latest_exported_artifacts(
        drive_export_root=drive_export_root,
        local_artifact_root=repo_root / 'artifacts',
        runtime_experiment_config=runtime_experiment_config,
    )
elif not summary_path.exists() and checkpoint_dir.exists() and not any(checkpoint_dir.glob('*.pt')):
    restored_export_run = restore_latest_exported_artifacts(
        drive_export_root=drive_export_root,
        local_artifact_root=repo_root / 'artifacts',
        runtime_experiment_config=runtime_experiment_config,
    )

if runtime_experiment_config.exists():
    experiment_config = runtime_experiment_config
    runtime_payload = yaml.safe_load(experiment_config.read_text(encoding='utf-8'))
    dataset_selection_active = any(
        value not in (None, [], '', {})
        for key, value in runtime_payload.get('dataset_selection', {}).items()
        if key != 'output_name'
    )
    selection_output_name = runtime_payload.get('dataset_selection', {}).get('output_name', 'unknown')
else:
    runtime_context = prepare_notebook_runtime_context(
        base_experiment_config=base_experiment_config,
        runtime_experiment_config=runtime_experiment_config,
        session_catalog_csv=repo_root / 'data' / 'allen_visual_behavior_neuropixels' / 'manifests' / 'session_catalog.csv',
        artifact_root=repo_root / 'artifacts',
        dataset_config=NOTEBOOK_DATASET,
        step_log_every=NOTEBOOK_STEP_LOG_EVERY,
    )
    experiment_config = runtime_context.experiment_config_path
    checkpoint_dir = runtime_context.checkpoint_dir
    summary_path = runtime_context.summary_path
    dataset_selection_active = runtime_context.dataset_selection_active
    selection_output_name = runtime_context.selection_output_name
    selected_session_count = runtime_context.selected_session_count

selected_checkpoint = resolve_notebook_checkpoint(summary_path=summary_path, checkpoint_dir=checkpoint_dir)
run_label = datetime.now().strftime('discover_run_%Y%m%d_%H%M%S')
drive_run_export_root = drive_export_root / run_label

config_source = 'existing runtime config' if runtime_experiment_config.exists() else 'generated from NOTEBOOK_DATASET'
print(f'Runtime config: {config_source} | selection active: {dataset_selection_active} | checkpoint: {selected_checkpoint.name}')
if restored_export_run is not None:
    print(f'Restored exported training run from: {restored_export_run}')

In [ ]:
paths_ok = verify_paths_exist({
    'experiment_config': experiment_config,
    'data_config': data_config,
    'checkpoint': selected_checkpoint,
    'drive_dataset_root': drive_data_root,
})
print(paths_ok)
assert all(paths_ok.values()), 'Missing discovery/validation inputs.'

if dataset_selection_active:
    reporter.begin('runtime selection', next_artifact='selected split manifest')
    run_streaming_command([
        'pcc-prepare-data',
        'materialize-runtime-selection',
        '--config', str(experiment_config),
        '--data-config', str(data_config),
    ], cwd=repo_root, step_log_every=NOTEBOOK_STEP_LOG_EVERY)
    reporter.finish('runtime selection')
else:
    print('Using the full canonical dataset. No runtime subset will be materialized.')

In [ ]:
reporter.begin('discovery', next_artifact='local discovery artifact + cluster summary')
%cd {repo_root}
resolved_discovery_split = None
last_discovery_error = None
for candidate_split in DISCOVERY_SPLIT_CANDIDATES:
    print(f'Trying discovery split: {candidate_split}')
    try:
        run_streaming_command([
            'pcc-discover',
            '--config', str(experiment_config),
            '--data-config', str(data_config),
            '--checkpoint', str(selected_checkpoint),
            '--split', candidate_split,
        ], cwd=repo_root, step_log_every=NOTEBOOK_STEP_LOG_EVERY)
        resolved_discovery_split = candidate_split
        break
    except subprocess.CalledProcessError as exc:
        last_discovery_error = exc
        output = exc.output or ''
        if output_indicates_missing_positive_labels(output):
            print(f'No positive target-label windows were found on split {candidate_split}; trying the next split.')
            continue
        raise
if resolved_discovery_split is None:
    if last_discovery_error is not None:
        raise last_discovery_error
    raise RuntimeError('Discovery did not complete and no split fallback succeeded.')
print('Resolved discovery split:', resolved_discovery_split)
reporter.finish('discovery')

In [ ]:
discovery_artifact = Path(selected_checkpoint).with_name(f"{Path(selected_checkpoint).stem}_{resolved_discovery_split}_discovery.json")
cluster_summary_json = discovery_artifact.with_name(f"{discovery_artifact.stem}_cluster_summary.json")
cluster_summary_csv = discovery_artifact.with_name(f"{discovery_artifact.stem}_cluster_summary.csv")

reporter.begin('validation', next_artifact='local validation summary json/csv')
%cd {repo_root}
run_streaming_command([
    'pcc-validate',
    '--config', str(experiment_config),
    '--data-config', str(data_config),
    '--checkpoint', str(selected_checkpoint),
    '--discovery-artifact', str(discovery_artifact),
], cwd=repo_root, step_log_every=NOTEBOOK_STEP_LOG_EVERY)
reporter.finish('validation')

## Results

Visualization and summary of the completed run.


In [ ]:
import json
import pandas as pd

# Define validation artifact paths (produced by the validation command above).
validation_json = discovery_artifact.with_name(f"{discovery_artifact.stem}_validation.json")
validation_csv  = discovery_artifact.with_name(f"{discovery_artifact.stem}_validation.csv")

# --- Cluster summary table ---
print('=== Cluster Summary ===')
cluster_df = pd.read_csv(cluster_summary_csv)
display(cluster_df)

# --- Per-cluster one-liners ---
with open(cluster_summary_json, 'r', encoding='utf-8') as fh:
    cluster_summary_payload = json.load(fh)

print(
    f"\nTotal clusters: {cluster_summary_payload['cluster_count']}  "
    f"Total candidates: {cluster_summary_payload['candidate_count']}"
)
for cluster in cluster_summary_payload.get('clusters', []):
    top_regions = ', '.join(r['value'] for r in cluster.get('top_regions', [])[:3]) or 'n/a'
    print(
        f"  Cluster {cluster['cluster_id']:>3}: "
        f"{cluster['candidate_count']} candidates, "
        f"{cluster['session_count']} sessions, "
        f"mean score {cluster['mean_score']:.3f}, "
        f"top regions: {top_regions}"
    )

# --- Validation summary ---
if validation_json.exists():
    print('\n=== Validation Summary ===')
    with open(validation_json, 'r', encoding='utf-8') as fh:
        validation_payload = json.load(fh)
    real_acc  = validation_payload.get('real_label_metrics', {}).get('probe_accuracy', float('nan'))
    shuf_acc  = validation_payload.get('shuffled_label_metrics', {}).get('probe_accuracy', float('nan'))
    stability = validation_payload.get('stability_summary', {}).get('stability_score', float('nan'))
    recurrence = validation_payload.get('recurrence_summary', {}).get('recurrence_rate', float('nan'))
    print(f'  Probe accuracy:  real={real_acc:.3f}  shuffled={shuf_acc:.3f}')
    print(f'  Stability score: {stability:.3f}')
    print(f'  Recurrence rate: {recurrence:.3f}')
    issues = validation_payload.get('provenance_issues', [])
    if issues:
        print(f'  Provenance issues ({len(issues)}):')
        for issue in issues:
            print(f'    - {issue}')
    display(pd.read_csv(validation_csv))
else:
    print('\nValidation artifacts not found — run the validation cell above first.')

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import Counter

with open(cluster_summary_json, 'r', encoding='utf-8') as fh:
    cluster_payload = json.load(fh)
clusters = cluster_payload.get('clusters', [])

val_payload = None
if validation_json.exists():
    with open(validation_json, 'r', encoding='utf-8') as fh:
        val_payload = json.load(fh)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Panel 1: Candidate counts per cluster
cluster_ids = [str(c['cluster_id']) for c in clusters]
cand_counts = [c['candidate_count'] for c in clusters]
axes[0].bar(cluster_ids, cand_counts, color='#4c72b0')
axes[0].set_title('Candidates per Cluster')
axes[0].set_xlabel('Cluster ID')
axes[0].set_ylabel('Candidate Count')
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=10))
for i, v in enumerate(cand_counts):
    axes[0].text(i, v + 0.3, str(v), ha='center', va='bottom', fontsize=8)

# Panel 2: Real vs shuffled probe accuracy
if val_payload is not None:
    real_acc = val_payload.get('real_label_metrics', {}).get('probe_accuracy', float('nan'))
    shuf_acc = val_payload.get('shuffled_label_metrics', {}).get('probe_accuracy', float('nan'))
    bars = axes[1].bar(['Real labels', 'Shuffled labels'], [real_acc, shuf_acc],
                       color=['#55a868', '#dd8452'], width=0.4)
    axes[1].set_title('Probe Accuracy: Real vs Shuffled')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_ylim(0, 1.1)
    axes[1].axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='Chance')
    axes[1].legend(fontsize=8)
    for bar, v in zip(bars, [real_acc, shuf_acc]):
        if v == v:
            axes[1].text(bar.get_x() + bar.get_width() / 2, v + 0.03, f'{v:.3f}', ha='center', fontsize=9)
else:
    axes[1].text(0.5, 0.5, 'Validation not run', ha='center', va='center',
                 transform=axes[1].transAxes, fontsize=11, color='gray')
    axes[1].set_title('Probe Accuracy: Real vs Shuffled')
    axes[1].axis('off')

# Panel 3: Top brain regions across all clusters
region_counter = Counter()
for cluster in clusters:
    for entry in cluster.get('top_regions', []):
        region_counter[entry['value']] += entry['count']

if region_counter:
    top_regions = region_counter.most_common(10)
    names  = [r[0] for r in reversed(top_regions)]
    counts = [r[1] for r in reversed(top_regions)]
    axes[2].barh(names, counts, color='#8172b2')
    axes[2].set_title('Top Brain Regions (all clusters)')
    axes[2].set_xlabel('Cumulative Candidate Count')
    for i, v in enumerate(counts):
        axes[2].text(v + 0.1, i, str(v), va='center', fontsize=8)
else:
    axes[2].text(0.5, 0.5, 'No region data', ha='center', va='center',
                 transform=axes[2].transAxes, fontsize=11, color='gray')
    axes[2].axis('off')

fig.suptitle('Discovery & Validation Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import json

with open(cluster_summary_json, 'r', encoding='utf-8') as fh:
    cluster_payload = json.load(fh)

cluster_count  = cluster_payload['cluster_count']
candidate_count = cluster_payload['candidate_count']

val_payload = None
if validation_json.exists():
    with open(validation_json, 'r', encoding='utf-8') as fh:
        val_payload = json.load(fh)

if val_payload is not None:
    real_acc   = val_payload.get('real_label_metrics', {}).get('probe_accuracy')
    shuf_acc   = val_payload.get('shuffled_label_metrics', {}).get('probe_accuracy')
    stability  = val_payload.get('stability_summary', {}).get('stability_score')
    recurrence = val_payload.get('recurrence_summary', {}).get('recurrence_rate')

    acc_str  = f'{real_acc * 100:.1f}%'  if real_acc   is not None else 'n/a'
    shuf_str = f'{shuf_acc * 100:.1f}%'  if shuf_acc   is not None else 'n/a'
    stab_str = f'{stability:.2f}'         if stability  is not None else 'n/a'
    rec_str  = f'{recurrence * 100:.1f}%' if recurrence is not None else 'n/a'

    is_structured = real_acc and shuf_acc and real_acc > shuf_acc + 0.05
    structure_word = 'structured' if is_structured else 'marginal or no'

    print(
        f'Discovery found {cluster_count} cluster(s) containing {candidate_count} candidate neural windows.\n'
        f'A linear probe trained on the real cluster labels achieved {acc_str} accuracy, '
        f'compared to {shuf_str} on shuffled labels, '
        f'suggesting the clusters capture {structure_word} label-related structure.\n'
        f'Cluster stability across bootstrap rounds scored {stab_str} '
        f'and the recurrence rate was {rec_str}.'
    )
else:
    print(
        f'Discovery found {cluster_count} cluster(s) containing {candidate_count} candidate neural windows.\n'
        f'Validation was not run — probe accuracy, stability, and recurrence metrics are not available.'
    )

In [ ]:
reporter.begin('export', next_artifact='Drive copy of local artifacts')
if drive_run_export_root.exists():
    shutil.rmtree(drive_run_export_root)
shutil.copytree(repo_artifacts_root, drive_run_export_root)
print('Exported local artifacts to:', drive_run_export_root)
reporter.finish('export')